# Meet the data

Every notebook in this gallery tells its story with the same fictional business:
**Fernway**, an online retailer of home and lifestyle goods with a paid membership,
**Fernway Plus**. The data covers 2024 and 2025 in seven tables that join on their
keys, as they would in a company's data warehouse.

This notebook introduces the tables. It draws no charts: those live in the plot
notebooks, each of which starts from a SQL query like the ones below and passes the
result straight to an easy-viz function.

Ten patterns are hidden in the data, each the kind of thing a default chart misses.
Every plot notebook ends with an *Insight story* that uncovers one of them. The
answer key, which also defines every column, is
[`data/INSIGHTS.md`](data/INSIGHTS.md); if you want to hunt for the patterns
yourself, leave it closed for now.

## Setup

The tables are CSV files in [`data/`](data/). This cell turns each one into a DuckDB
view named after the file, so the queries read like warehouse SQL. Every gallery
notebook starts with the same cell (plus `import easy_viz as ev`).

In [1]:
from pathlib import Path

import duckdb

db = duckdb.connect()
for csv in sorted(Path("data").glob("*.csv")):
    db.sql(f"CREATE VIEW {csv.stem} AS FROM '{csv.as_posix()}'")

db.sql("SHOW TABLES").df()

,name
0,checkout_test
1,churn_scores
2,customers
3,daily_metrics
4,listings
5,orders
6,products


## Customers

One row per active customer: who they are (region, segment, acquisition channel,
age, income, membership plan) and how they behaved over the last year, in three
groups of columns: engagement, spending and service.

In [2]:
db.sql("FROM customers LIMIT 5").df()

,customer_id,signup_date,region,segment,acquisition_channel,age,annual_income,plan,sessions_per_month,pages_per_session,app_opens_per_month,email_click_rate,orders_12m,spend_12m,items_per_order,support_tickets_12m,returns_12m,late_deliveries_12m,satisfaction_score
0,C00001,2022-12-12,South,Consumer,Referral,41,32700,Standard,14.3,5.2,8,0.0938,11,1330.79,2.91,2,1,1,7
1,C00002,2023-03-26,East,Consumer,Organic search,56,52600,Standard,27.0,6.9,24,0.3211,3,139.74,1.33,2,1,1,5
2,C00003,2024-03-17,West,Consumer,Referral,65,55000,Standard,12.6,5.1,10,0.1144,1,15.99,1.00,2,1,1,7
3,C00004,2024-06-01,East,Consumer,Organic search,45,50300,Plus,5.4,2.7,5,0.0429,3,549.86,1.67,1,0,0,8
4,C00005,2023-01-05,East,Consumer,Organic search,48,65500,Plus,8.1,4.8,3,0.0985,6,1621.81,2.33,1,0,1,8


## Products

One row per product: its category, what it costs Fernway and its current list price.

In [3]:
db.sql("FROM products LIMIT 5").pl()

product_id,product_name,category,unit_cost,list_price
str,str,str,f64,f64
"""P0001""","""Meadow Armchair""","""Furniture""",138.1,228.99
"""P0002""","""Alder Desk""","""Furniture""",198.25,331.99
"""P0003""","""Juniper Coffee Table""","""Furniture""",290.81,638.99
"""P0004""","""Slate Coffee Table""","""Furniture""",126.99,226.99
"""P0005""","""Juniper Desk""","""Furniture""",43.42,106.99


## Orders

The heart of the business: one row per order, with one product line each. It records
who bought what, from which warehouse, at what price and discount, the revenue and
profit, and how many days delivery took.

In [4]:
db.sql("FROM orders LIMIT 5").df()

,order_id,order_date,customer_id,product_id,warehouse,quantity,unit_price,discount,revenue,profit,delivery_days
0,O000001,2024-01-01,C01167,P0287,North hub,5,29.46,0.1,132.57,38.92,1.9
1,O000002,2024-01-01,C02930,P0090,South hub,3,40.17,0.1,108.46,50.56,6.1
2,O000003,2024-01-01,C02521,P0075,South hub,1,22.31,0.0,22.31,1.57,4.2
3,O000004,2024-01-01,C01458,P0171,North hub,2,12.49,0.0,24.98,7.92,2.6
4,O000005,2024-01-01,C01245,P0031,North hub,2,182.13,0.0,364.26,129.44,1.9


## Daily metrics

One row per day: site visits (`sessions`), orders, revenue, conversion rate and
average order value. Orders and revenue are the daily totals of the `orders` table,
so the two always agree.

In [5]:
db.sql("FROM daily_metrics ORDER BY date LIMIT 5").pl()

date,sessions,orders,revenue,conversion_rate,avg_order_value
date,i64,i64,f64,f64,f64
2024-01-01,1738,49,5768.88,0.0282,117.73
2024-01-02,1552,51,5422.41,0.0329,106.32
2024-01-03,1658,49,5567.07,0.0296,113.61
2024-01-04,1534,38,5275.45,0.0248,138.83
2024-01-05,1537,38,4803.74,0.0247,126.41


## Listings

One row per product featured in a regional web storefront in Q4 2025: its price, how
many people viewed its page and how many of them bought.

In [6]:
db.sql("FROM listings LIMIT 5").df()

,product_id,region,price,page_views,purchases,conversion_rate
0,P0001,North,228.99,6937,253,0.0365
1,P0001,West,228.99,7258,144,0.0198
2,P0002,West,331.99,2969,60,0.0202
3,P0003,North,638.99,3583,154,0.0430
4,P0003,West,638.99,2702,68,0.0252


## Churn scores

One row per Fernway Plus member: the churn probability a model gave them on
2025-09-30, and whether they actually cancelled in the 90 days that followed.

In [7]:
db.sql("FROM churn_scores LIMIT 5").pl()

customer_id,score_date,churn_probability,churned_90d
str,date,f64,i64
"""C00004""",2025-09-30,0.0478,0
"""C00005""",2025-09-30,0.0111,0
"""C00009""",2025-09-30,0.039,0
"""C00014""",2025-09-30,0.9228,0
"""C00016""",2025-09-30,0.0752,0


## Checkout test

One row per checkout in an A/B/C test of the checkout page: the variant shown, the
device, and the value of the order. These checkouts are a separate experiment, not
part of `orders`.

In [8]:
db.sql("FROM checkout_test LIMIT 5").df()

,checkout_id,variant,device,order_value
0,K0001,C,Mobile,133.27
1,K0002,A,Desktop,127.18
2,K0003,C,Mobile,50.87
3,K0004,C,Desktop,37.86
4,K0005,A,Mobile,38.47


## How the tables join

`orders` joins `customers` on `customer_id` and `products` on `product_id`;
`listings` joins `products`, and `churn_scores` joins `customers`. A typical gallery
query combines them and aggregates, like this one: revenue per region and membership
plan in 2025.

Its result has one row per region and plan, with the series stacked in the `plan`
column. That is the shape a `GROUP BY` naturally returns, and easy-viz plots it as
it is: `color="plan"` would draw one series per plan. No reshaping in between.

In [9]:
db.sql("""
    SELECT c.region, c.plan, round(sum(o.revenue)) AS revenue
    FROM orders AS o
    JOIN customers AS c USING (customer_id)
    WHERE year(o.order_date) = 2025
    GROUP BY c.region, c.plan
    ORDER BY c.region, c.plan
""").df()

,region,plan,revenue
0,East,Plus,312880.0
1,East,Standard,229997.0
2,North,Plus,260648.0
3,North,Standard,219080.0
4,South,Plus,252782.0
5,South,Standard,199520.0
6,West,Plus,217642.0
7,West,Standard,182815.0


## Next

Pick a plot from the [gallery index](README.md). Each notebook starts from a query on
these tables and walks from the one-line default chart to the chart that exposes one
of the hidden patterns.